In [1]:
import os
os.chdir("/kpsort")
import cv2
from IPython.display import display, Image, clear_output
from tools.kpsort import Sort
from tools.loadpkl import *
from ultralytics import YOLO
import numpy as np
import pickle
import math
import time

In [2]:
def imshow(img):
    _, buf = cv2.imencode(".jpg", img)
    display(Image(data=buf.tobytes()))
    clear_output(wait=True)

In [3]:
MODE_SAVE = 0
MODE_SHOW = 1

mode = MODE_SHOW

path_csv = "sources/out_DLC/outDLC_dlcrnetms5_bee0612Jun12shuffle1_200000_el.csv"
path_pkl = "sources/out_DLC/outDLC_dlcrnetms5_bee0612Jun12shuffle1_200000_full.pickle"
with open(path_pkl, "rb") as file:
    data_pkl: dict = pickle.load(file)
data_csv = load_csv(path_csv)
color_map = iter(gen_random_colors(10000, 334))

model = YOLO("best2.pt", task="predict")
cap = cv2.VideoCapture("sources/0615.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')
video = cv2.VideoWriter("output/video4.mp4",fourcc, cap.get(cv2.CAP_PROP_FPS), size)

mot_tracker = Sort(iou_threshold=0.001)

In [4]:
c = 0
colors = dict()
while True:
	st = time.time()
	#if c == 2: break
	success, frame = cap.read()
	if success:
		individuals = assemble_w_yolo(model, frame, data_pkl, data_csv, c)
		for individual in individuals:
			for i in range(0, len(individual) - 1, 2):
				if math.isnan(individual[i]): continue
				cv2.circle(frame, (int(individual[i]), int(individual[i + 1])), 5, (0, 255, 0), 5)
		trackers = mot_tracker.update(individuals)
		#print(trackers)
		for d in trackers:
			d = d.astype(np.int32)
			if np.any(d < 0): continue
			mask = str(bin(int(d[6])))[2:].zfill(6)
			if d[7] not in colors:
				colors[d[7]] = next(color_map)
			for i in range(0, len(d), 2):
				if i > 4: break
				if mask[i] != "1":
					cv2.circle(frame, (d[i], d[i + 1]), 5, colors[d[7]], 5)
			cv2.putText(frame, str(d[7]), (d[0], d[1]), cv2.FONT_HERSHEY_PLAIN, 1.0, colors[d[7]], 1, cv2.LINE_AA)
		c += 1
		imshow(frame)
		elapsed_time = time.time() - st
		if elapsed_time < 1 / fps:
			time.sleep(1 / fps - elapsed_time)
		#time.sleep(0.1)
	else: 
		break

KeyboardInterrupt: 